### Udemy scraper
# 🎓 Udemy Course Analyzer

A tool that scrapes Udemy course content using Playwright and feeds it to a 
local LLM to help you decide whether a course is worth buying — based on your 
personal learning goals.

## 🚀 What It Does

- Scrapes Udemy course curriculum (lectures, sections, descriptions) via Playwright
- Feeds the scraped content to a local LLM (via Ollama)
- Returns an honest, structured analysis of the course
- Helps you make smarter purchase decisions before spending money



### Imports


In [ ]:
from openai import OpenAI
import json
import sys
import subprocess



### base_variables

In [ ]:
MODEL = "llama3.1:8b"
ollama_base_url = "http://localhost:11434/v1"
client = OpenAI(base_url=ollama_base_url,api_key="ollama_apikey")
# change tis to analyze any course on udemy
udemy_course = "https://www.udemy.com/course/mastering-gpu-parallel-programming-with-cuda/"

### sscrape and import json data here and converison to text 

In [ ]:
subprocess.run([sys.executable, "run_scraper.py", udemy_course], check=True)



with open("udemy_outline.json", "r") as f:
    course = json.load(f)

text = f"Course: {course['course']}\n\n"

for section in course['sections']:
    text += f"### {section['section']}\n"
    for lecture in section['lectures']:
        text += f"  - {lecture}\n"
    text += "\n"

with open("udemy_outline.txt", "w") as f:
    f.write(text)

print("Done! Saved to udemy_outline.txt")

with open("udemy_outline.txt", "r") as f:
    course_material_str = f.read()



### Prompts

In [ ]:
# ✅ System prompt = Fixed with exact termination marker matching the stop token
system_prompt = """``You are a professional Udemy course analyst. Your task is to analyze the course material provided and determine if it aligns with the user's goal.
   
    """

goal = "cuda engineer"

print(course_material_str)  # Print first 500 chars to verify

# Clean user prompt: Only raw structural context, no conversational text or instructions
user_prompt = f"""Analyze this Udemy course for someone whose goal is: {goal}

{course_material_str}


Now output the verdict block. Start with ### 📊 Course Verdict:"""

print(user_prompt)

### pass to model

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ],
     # 1. CRITICAL: Drops randomness to zero
    
)

print(response.choices[0].message.content)